In [53]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


data_dir = Path("data/features/features_with_ME")
files = list(data_dir.glob("*.csv"))
stock_data = {f.stem.replace("_features_ME", ""): pd.read_csv(f) for f in files}



def make_direction_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors = "coerce")

    for d in [5,10,20]:
        ma_col = f"MA{d}"
        ma_next = df[ma_col].shift(-1)
        dir_col = f"MA{d}_dir"
        df[dir_col] = np.where(ma_next > df[ma_col], 1 , -1)

    df = df.iloc[:-1].reset_index(drop = True)
    return df

labeled_data = {ticker: make_direction_labels(df) for ticker, df in stock_data.items()}

In [54]:
# Now we need to split the data into training and testing sets
# But since we have 3 different targets, we need to split the data for each target
# And since this is the time series data, we need to split the data by the index, not the random split

targets_dir = ["MA5_dir", "MA10_dir", "MA20_dir"]

split_data = {}

ME_cols = ["FFR_Level", "CPI_YoY", "UNRATE_Level", "YC_Slope_10Y3M", "PMI_Level"]
ME_weight = 50.0

for ticker, df in labeled_data.items():

    df = df.copy()
    df = df.sort_values("Date")


    exclude_cols = ["Date"] + targets_dir

    split_idx = int(len(df) * 0.8)

    X_all = df.drop(columns=exclude_cols, errors="ignore")  
    X_train_raw = X_all.iloc[:split_idx].copy()
    X_test_raw  = X_all.iloc[split_idx:].copy()

    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), 
                           columns=X_train_raw.columns, index=X_train_raw.index)
    X_test  = pd.DataFrame(scaler.transform(X_test_raw), 
                           columns=X_test_raw.columns, index=X_test_raw.index)
    
    for col in ME_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col] * ME_weight
            X_test[col]  = X_test[col] * ME_weight

    split_data[ticker] = {"X_train": X_train, "X_test": X_test}
    for t in targets_dir:
        y_all = df[t]
        y_train = y_all.iloc[:split_idx]
        y_test  = y_all.iloc[split_idx:]
        split_data[ticker][t] = {"y_train": y_train, "y_test": y_test}

In [55]:
# Running the Decision Tree Regression
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results= []
models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    models[ticker] = {}

    for target in targets_dir:
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        reg = DecisionTreeRegressor(
            random_state=42,
            max_depth=None,
            min_samples_leaf=5,
            criterion="squared_error" 
        )

        reg.fit(X_train, y_train)

        y_pred_cont = reg.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)  

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)


        results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  
            "Model": "DecisionTreeClassifier",
            "MAE": mae,                    
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        models[ticker][target] = reg

res_df = pd.DataFrame(results).sort_values(["Ticker", "Target"])

res_df["Target"] = pd.Categorical(res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
res_df = res_df.sort_values(["Ticker","Target"])

table = (
    res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)

print(table.to_string())

                                           MAE       MSE      RMSE        R2
Ticker Target Model                                                         
AAPL   MA5    DecisionTreeClassifier  0.514085  1.028169  1.013987 -0.036859
       MA10   DecisionTreeClassifier  0.373239  0.746479  0.863990  0.243922
       MA20   DecisionTreeClassifier  0.295775  0.591549  0.769122  0.387931
ADBE   MA5    DecisionTreeClassifier  0.380282  0.760563  0.872103  0.239097
       MA10   DecisionTreeClassifier  0.338028  0.676056  0.822226  0.312903
       MA20   DecisionTreeClassifier  0.225352  0.450704  0.671345  0.514530
AMD    MA5    DecisionTreeClassifier  0.732394  1.464789  1.210285 -0.467409
       MA10   DecisionTreeClassifier  0.450704  0.901408  0.949425  0.097473
       MA20   DecisionTreeClassifier  0.140845  0.281690  0.530745  0.716906
CRM    MA5    DecisionTreeClassifier  0.443662  0.887324  0.941979  0.111971
       MA10   DecisionTreeClassifier  0.330986  0.661972  0.813617  0.334729

In [52]:
# Running the SVM
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


svm_results = []
svm_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    svm_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        svm = SVR(kernel="linear", C=1.0, epsilon=0.1)
        svm.fit(X_train, y_train)

        y_pred_cont = svm.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        svm_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""), 
            "Model": "SVM",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        svm_models[ticker][target] = svm

svm_res_df = pd.DataFrame(svm_results)
svm_res_df["Target"] = pd.Categorical(svm_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
svm_table = (
    svm_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(svm_table.to_string())

KeyboardInterrupt: 

In [56]:
# Runnding Bagging 

from sklearn.ensemble import BaggingRegressor

bag_results = []
bag_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    bag_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        base = DecisionTreeRegressor(random_state=42, min_samples_leaf=5)
        model = BaggingRegressor(
            estimator=base,        # scikit-learn >= 1.2
            n_estimators=300,
            bootstrap=True,
            n_jobs=-1,
            random_state=42
        )
        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        bag_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "BaggingRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        bag_models[ticker][target] = model

bag_res_df = pd.DataFrame(bag_results)
bag_res_df["Target"] = pd.Categorical(bag_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
bag_table = (
    bag_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(bag_table.to_string())

                                     MAE       MSE      RMSE        R2
Ticker Target Model                                                   
AAPL   MA5    BaggingRegressor  0.323944  0.647887  0.804914  0.346637
       MA10   BaggingRegressor  0.260563  0.521127  0.721891  0.472172
       MA20   BaggingRegressor  0.211268  0.422535  0.650027  0.562808
ADBE   MA5    BaggingRegressor  0.309859  0.619718  0.787222  0.380005
       MA10   BaggingRegressor  0.218310  0.436620  0.660772  0.556250
       MA20   BaggingRegressor  0.169014  0.338028  0.581402  0.635897
AMD    MA5    BaggingRegressor  0.323944  0.647887  0.804914  0.350954
       MA10   BaggingRegressor  0.260563  0.521127  0.721891  0.478226
       MA20   BaggingRegressor  0.126761  0.253521  0.503509  0.745215
CRM    MA5    BaggingRegressor  0.359155  0.718310  0.847532  0.281120
       MA10   BaggingRegressor  0.218310  0.436620  0.660772  0.561204
       MA20   BaggingRegressor  0.190141  0.380282  0.616670  0.619699
MSFT  

In [57]:
# Random forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_results = []
rf_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    rf_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        model = RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
            bootstrap=True
        )
        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        rf_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "RandomForestRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        rf_models[ticker][target] = model

rf_res_df = pd.DataFrame(rf_results)
rf_res_df["Target"] = pd.Categorical(rf_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
rf_table = (
    rf_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(rf_table.to_string())

                                          MAE       MSE      RMSE        R2
Ticker Target Model                                                        
AAPL   MA5    RandomForestRegressor  0.366197  0.732394  0.855800  0.261415
       MA10   RandomForestRegressor  0.274648  0.549296  0.741145  0.443641
       MA20   RandomForestRegressor  0.211268  0.422535  0.650027  0.562808
ADBE   MA5    RandomForestRegressor  0.316901  0.633803  0.796117  0.365914
       MA10   RandomForestRegressor  0.211268  0.422535  0.650027  0.570565
       MA20   RandomForestRegressor  0.183099  0.366197  0.605142  0.605556
AMD    MA5    RandomForestRegressor  0.316901  0.633803  0.796117  0.365064
       MA10   RandomForestRegressor  0.281690  0.563380  0.750587  0.435920
       MA20   RandomForestRegressor  0.126761  0.253521  0.503509  0.745215
CRM    MA5    RandomForestRegressor  0.366197  0.732394  0.855800  0.267024
       MA10   RandomForestRegressor  0.211268  0.422535  0.650027  0.575359
       MA20 

In [40]:
# Ada boost

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ada_results = []
ada_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    ada_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        weak = DecisionTreeRegressor(max_depth=3, random_state=42)

        try:
            model = AdaBoostRegressor(
                estimator=weak,           # scikit-learn >= 1.2
                n_estimators=300,
                learning_rate=0.1,
                random_state=42
            )
        except TypeError:
            model = AdaBoostRegressor(
                base_estimator=weak,      # scikit-learn < 1.2
                n_estimators=300,
                learning_rate=0.1,
                random_state=42
            )

        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        ada_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "AdaBoostRegressor(Depth3Tree)",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        ada_models[ticker][target] = model

ada_res_df = pd.DataFrame(ada_results)
ada_res_df["Target"] = pd.Categorical(ada_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
ada_table = (
    ada_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(ada_table.to_string())

                                                  MAE       MSE      RMSE        R2
Ticker Target Model                                                                
AAPL   MA5    AdaBoostRegressor(Depth3Tree)  0.302817  0.605634  0.778225  0.389247
       MA10   AdaBoostRegressor(Depth3Tree)  0.295775  0.591549  0.769122  0.400844
       MA20   AdaBoostRegressor(Depth3Tree)  0.232394  0.464789  0.681754  0.519089
ADBE   MA5    AdaBoostRegressor(Depth3Tree)  0.316901  0.633803  0.796117  0.365914
       MA10   AdaBoostRegressor(Depth3Tree)  0.253521  0.507042  0.712069  0.484677
       MA20   AdaBoostRegressor(Depth3Tree)  0.183099  0.366197  0.605142  0.605556
AMD    MA5    AdaBoostRegressor(Depth3Tree)  0.323944  0.647887  0.804914  0.350954
       MA10   AdaBoostRegressor(Depth3Tree)  0.260563  0.521127  0.721891  0.478226
       MA20   AdaBoostRegressor(Depth3Tree)  0.119718  0.239437  0.489323  0.759370
CRM    MA5    AdaBoostRegressor(Depth3Tree)  0.316901  0.633803  0.796117  0

In [41]:
# Cat boost
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool


cat_results = []
cat_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    cat_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        train_pool = Pool(X_train, y_train)
        test_pool  = Pool(X_test,  y_test)

        model = CatBoostRegressor(
            iterations=300,
            depth=6,
            learning_rate=0.1,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

        model.fit(train_pool)

        y_pred_cont = model.predict(test_pool)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        cat_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "CatBoostRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        cat_models[ticker][target] = model

cat_res_df = pd.DataFrame(cat_results)
cat_res_df["Target"] = pd.Categorical(cat_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
cat_table = (
    cat_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(cat_table.to_string())

                                      MAE       MSE      RMSE        R2
Ticker Target Model                                                    
AAPL   MA5    CatBoostRegressor  0.380282  0.760563  0.872103  0.233008
       MA10   CatBoostRegressor  0.274648  0.549296  0.741145  0.443641
       MA20   CatBoostRegressor  0.253521  0.507042  0.712069  0.475369
ADBE   MA5    CatBoostRegressor  0.330986  0.661972  0.813617  0.337733
       MA10   CatBoostRegressor  0.232394  0.464789  0.681754  0.527621
       MA20   CatBoostRegressor  0.211268  0.422535  0.650027  0.544872
AMD    MA5    CatBoostRegressor  0.316901  0.633803  0.796117  0.365064
       MA10   CatBoostRegressor  0.281690  0.563380  0.750587  0.435920
       MA20   CatBoostRegressor  0.098592  0.197183  0.444053  0.801834
CRM    MA5    CatBoostRegressor  0.380282  0.760563  0.872103  0.238833
       MA10   CatBoostRegressor  0.218310  0.436620  0.660772  0.561204
       MA20   CatBoostRegressor  0.197183  0.394366  0.627986  0

In [43]:
# xbg boost

from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

xgb_results = []
xgb_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    xgb_models[ticker] = {}

    for target in targets_dir:
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        y_train_bin = (y_train == 1).astype(int)
        y_test_bin  = (y_test == 1).astype(int)

        xgb = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            random_state=42
        )

        xgb.fit(X_train, y_train_bin)

        y_pred_bin = xgb.predict(X_test)

        y_pred = np.where(y_pred_bin == 1, 1, -1)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        xgb_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),
            "Model": "XGBoostClassifier",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        xgb_models[ticker][target] = xgb

xgb_df = pd.DataFrame(xgb_results)
print(xgb_df.to_string())


   Ticker Target              Model       MAE       MSE      RMSE        R2
0    AAPL    MA5  XGBoostClassifier  0.366197  0.732394  0.855800  0.261415
1    AAPL   MA10  XGBoostClassifier  0.295775  0.591549  0.769122  0.400844
2    AAPL   MA20  XGBoostClassifier  0.225352  0.450704  0.671345  0.533662
3    ADBE    MA5  XGBoostClassifier  0.302817  0.605634  0.778225  0.394096
4    ADBE   MA10  XGBoostClassifier  0.295775  0.591549  0.769122  0.398790
5    ADBE   MA20  XGBoostClassifier  0.197183  0.394366  0.627986  0.575214
6     AMD    MA5  XGBoostClassifier  0.323944  0.647887  0.804914  0.350954
7     AMD   MA10  XGBoostClassifier  0.246479  0.492958  0.702109  0.506430
8     AMD   MA20  XGBoostClassifier  0.133803  0.267606  0.517306  0.731061
9     CRM    MA5  XGBoostClassifier  0.401408  0.802817  0.896001  0.196546
10    CRM   MA10  XGBoostClassifier  0.218310  0.436620  0.660772  0.561204
11    CRM   MA20  XGBoostClassifier  0.225352  0.450704  0.671345  0.549273
12   MSFT   

In [45]:
# lgb boost

from lightgbm import LGBMClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

lgb_results = []
lgb_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    lgb_models[ticker] = {}

    for target in targets_dir:

        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        y_train_bin = (y_train == 1).astype(int)
        y_test_bin  = (y_test == 1).astype(int)

        lgb = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            max_depth=-1,         
            num_leaves=31,        
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose = -1
        )

        lgb.fit(X_train, y_train_bin)

        y_pred_bin = lgb.predict(X_test)

        y_pred = np.where(y_pred_bin == 1, 1, -1)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        lgb_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),
            "Model": "LightGBMClassifier",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        lgb_models[ticker][target] = lgb

lgb_df = pd.DataFrame(lgb_results)
print(lgb_df.to_string())


   Ticker Target               Model       MAE       MSE      RMSE        R2
0    AAPL    MA5  LightGBMClassifier  0.366197  0.732394  0.855800  0.261415
1    AAPL   MA10  LightGBMClassifier  0.260563  0.521127  0.721891  0.472172
2    AAPL   MA20  LightGBMClassifier  0.239437  0.478873  0.692007  0.504516
3    ADBE    MA5  LightGBMClassifier  0.316901  0.633803  0.796117  0.365914
4    ADBE   MA10  LightGBMClassifier  0.302817  0.605634  0.778225  0.384476
5    ADBE   MA20  LightGBMClassifier  0.232394  0.464789  0.681754  0.499359
6     AMD    MA5  LightGBMClassifier  0.323944  0.647887  0.804914  0.350954
7     AMD   MA10  LightGBMClassifier  0.253521  0.507042  0.712069  0.492328
8     AMD   MA20  LightGBMClassifier  0.119718  0.239437  0.489323  0.759370
9     CRM    MA5  LightGBMClassifier  0.338028  0.676056  0.822226  0.323407
10    CRM   MA10  LightGBMClassifier  0.246479  0.492958  0.702109  0.504585
11    CRM   MA20  LightGBMClassifier  0.232394  0.464789  0.681754  0.535188

In [46]:
import pandas as pd

tables = [
    table,
    svm_table,
    bag_table,
    rf_table,
    ada_table,
    cat_table,
    xgb_df,
    lgb_df
]

clean_tables = []
for t in tables:
    df = t.reset_index()         
    df.columns = df.columns.astype(str)
    clean_tables.append(df)


combined_df = pd.concat(clean_tables, ignore_index=True)

model_avg = (
    combined_df
    .groupby("Model")[["MAE", "MSE", "RMSE", "R2"]]
    .mean()
    .sort_values("MAE")
)

print(model_avg.to_string())

                                    MAE       MSE      RMSE        R2
Model                                                                
AdaBoostRegressor(Depth3Tree)  0.246479  0.492958  0.693754  0.497638
BaggingRegressor               0.256455  0.512911  0.707509  0.477245
RandomForestRegressor          0.268486  0.536972  0.722828  0.452785
XGBoostClassifier              0.274354  0.548709  0.732244  0.441071
LightGBMClassifier             0.281984  0.563967  0.743325  0.425241
CatBoostRegressor              0.287852  0.575704  0.747595  0.412806
SVM                            0.306925  0.613850  0.772424  0.372515
DecisionTreeClassifier         0.414319  0.828638  0.892056  0.156586


In [47]:
import numpy as np
import pandas as pd

# -------------------------------
# 1. Backtest Function
# -------------------------------
def backtest(prices, signals, initial_capital=100000):
    position = 0
    cash = initial_capital
    shares = 0
    
    for i in range(len(signals)):
        if signals[i] == 1 and position == 0:
            shares = cash / prices[i]
            cash = 0
            position = 1
        elif signals[i] == -1 and position == 1:
            cash = shares * prices[i]
            shares = 0
            position = 0
    
    final_value = cash + shares * prices[-1]
    return final_value


# -------------------------------
# 2. Collect predictions from ALL models
# -------------------------------
all_models = {
    "DecisionTree": models,
    "SVM": svm_models,
    "Bagging": bag_models,
    "RandomForest": rf_models,
    "AdaBoost": ada_models,
    "CatBoost": cat_models,
    "XGBoost": xgb_models,
    "LightGBM": lgb_models
}

targets = ["MA5_dir", "MA10_dir", "MA20_dir"]


# -------------------------------
# 3. Run backtesting for each model × each MA target × each stock
# -------------------------------
results = []

for ticker in split_data.keys():

    df = labeled_data[ticker]
    test_len = len(split_data[ticker]["X_test"])
    close_prices = df["Close"].iloc[-test_len:].values

    for target in targets:
        for model_name, model_dict in all_models.items():

            model = model_dict[ticker][target]
            y_pred_cont = model.predict(split_data[ticker]["X_test"])
            signals = np.where(y_pred_cont > 0, 1, -1)

            final_value = backtest(close_prices, signals)
            total_return = (final_value - 100000) / 100000 * 100

            results.append({
                "Ticker": ticker,
                "Target": target.replace("_dir",""),  
                "Model": model_name,
                "Return%": round(total_return, 2)
            })

backtest_df = pd.DataFrame(results)


# -------------------------------
# 4. Pivot Tables (MA5, MA10, MA20)
# -------------------------------
for ma in ["MA5", "MA10", "MA20"]:
    print(f"\n===================== {ma} Result =====================\n")
    
    tmp = backtest_df[backtest_df["Target"] == ma]
    
    pivot = tmp.pivot_table(
        index="Ticker",    # 종목
        columns="Model",   # 모델들
        values="Return%",  # 수익률
        aggfunc="mean"
    )
    
    display(pivot)



===================== MA5 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,6.11,4.23,9.53,13.86,3.50,5.22,5.76,0.39
ADBE,-6.65,-8.72,-13.45,10.19,-13.34,-8.94,-5.48,-12.71
AMD,10.84,13.62,17.34,31.33,16.65,11.34,7.98,15.42
CRM,4.10,-1.19,-10.81,-7.14,3.91,1.13,13.17,2.72
MSFT,8.08,12.35,5.30,9.05,-3.59,11.54,17.53,8.41
NOW,-12.69,-6.39,-17.77,-20.32,-28.37,1.65,-12.58,-15.90
NVDA,32.43,47.58,1.83,40.05,35.38,35.10,14.87,33.50
ORCL,70.73,51.45,68.64,89.00,38.81,40.96,105.78,46.67



===================== MA10 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,-14.43,-0.70,-7.37,-2.94,-10.35,-2.06,-10.95,-5.37
ADBE,-15.80,-8.81,-10.65,-11.47,-29.46,-9.10,-13.07,-24.58
AMD,15.82,12.79,1.38,2.23,4.77,14.38,9.61,11.38
CRM,21.15,23.98,20.12,17.77,18.92,29.00,0.97,23.56
MSFT,18.06,15.21,10.13,11.79,10.75,16.72,18.28,20.56
NOW,33.93,14.53,17.52,-9.08,-4.36,7.25,34.05,-0.27
NVDA,-3.49,0.35,-0.80,6.43,-4.92,-4.09,13.34,-12.01
ORCL,178.78,140.48,78.47,64.94,138.61,143.78,59.16,128.80



===================== MA20 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,-1.80,2.55,1.66,10.60,3.51,4.64,19.57,1.67
ADBE,-12.60,-7.98,-7.16,-6.80,-14.11,-7.18,-13.61,-7.86
AMD,5.56,10.33,29.19,33.01,11.65,22.58,19.57,17.17
CRM,0.98,0.32,3.56,-25.03,-2.67,3.39,10.41,-3.02
MSFT,-0.04,-0.27,-0.12,11.88,4.59,1.28,1.29,-0.05
NOW,29.42,28.63,29.44,1.37,23.70,30.26,22.97,29.69
NVDA,10.87,7.76,43.66,25.33,20.38,23.85,-3.17,12.61
ORCL,62.23,66.28,70.07,66.99,68.96,71.94,88.65,68.84
